# ReaVector AI — Kaggle main GPU (T4 ×2)

Personal / owner use only. Pulls one queued job from the Render bridge, runs it on a pinned T4, uploads the GLB back. No public serving.

**GPU 0** = images / LoRA / textures · **GPU 1** = video / 3D.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/rizbiehelp-sys/reavectors.git
%cd reavectors
!pip -q install numpy pillow trimesh requests rembg onnxruntime
!pip -q install diffusers transformers accelerate safetensors


## Bridge settings
Paste your `AI_BRIDGE_KEY` (from Render → Environment). Keep it in this notebook only — never commit it.

In [ ]:
import os, json, requests
BRIDGE = os.environ.get('BRIDGE_URL', 'https://reavectors.onrender.com')
KEY = os.environ.get('BRIDGE_KEY', 'PASTE_AI_BRIDGE_KEY_HERE')
H = {'X-Bridge-Key': KEY}
def pending():
    r = requests.get(f'{BRIDGE}/api/ai/bridge/jobs', params={'status':'pending'}, headers=H, timeout=30)
    r.raise_for_status(); return r.json()['jobs']
def claim(jid):
    return requests.post(f'{BRIDGE}/api/ai/bridge/jobs/{jid}/claim', headers=H, timeout=30).json()
print('bridge =', BRIDGE, '| key set =', KEY != 'PASTE_AI_BRIDGE_KEY_HERE')
pending()

## Install the 3D model (TripoSR by default)
For SF3D swap the clone/requirements. Hunyuan3D-2.1 (~24 GB) will not fit a T4.

In [ ]:
!git clone https://github.com/VAST-AI-Research/TripoSR /opt/TripoSR
!pip -q install -r /opt/TripoSR/requirements.txt
os.environ['PYTHONPATH'] = '/opt/TripoSR:' + os.environ.get('PYTHONPATH','')

## Run the next queued job
Downloads the job's input image, runs on GPU 1 (3D), uploads the GLB. The website can then load the artifact URL.

In [ ]:
import subprocess, json as _json, time
EXT = {'image2_3d':'.glb','text2image':'.png','image_edit':'.png','texture':'.png','text2video':'.mp4','image2video':'.mp4'}
DEADLINE = time.time() + int(os.environ.get('RV_KERNEL_SECONDS', '2700'))
MAX_JOBS = int(os.environ.get('RV_MAX_JOBS', '20'))
done = 0
for _ in range(MAX_JOBS):
    if time.time() > DEADLINE:
        print('time budget reached'); break
    jobs = pending()
    if not jobs:
        print('queue empty'); break
    j = claim(jobs[0]['id'])
    if not j:
        continue
    print('claim', j.get('model'), j.get('task'), flush=True)
    url = j.get('input_url')
    if url:
        inp = '/kaggle/working/' + os.path.basename(url.split('?')[0])
        open(inp, 'wb').write(requests.get(url, timeout=120).content)
    else:
        inp = '/kaggle/working/prompt.txt'
        open(inp, 'w').write((j.get('params') or {}).get('prompt', 'a product photo'))
    ext = EXT.get(j.get('task'), '.bin')
    out = '/kaggle/working/result' + ext
    gpu = '1' if j.get('model') in ('triposr','sf3d','hunyuan3d','hunyuan3d-mini','ltx-video','wan2.1') else '0'
    params = _json.dumps(j.get('params') or {})
    r = subprocess.run(['python','scripts/gpu_job.py','--gpu',gpu,'--model',j['model'],'--image',inp,'--out',out,'--params',params], capture_output=True, text=True)
    print((r.stdout or '')[-800:], (r.stderr or '')[-800:], flush=True)
    if os.path.exists(out):
        up = requests.post(f"{BRIDGE}/api/ai/bridge/jobs/{j['id']}/result", headers=H,
                           files={'file': open(out,'rb')}, data={'filename':'result'+ext}, timeout=300)
        print('upload', up.status_code, flush=True)
        done += 1
        os.remove(out)
    else:
        requests.post(f"{BRIDGE}/api/ai/bridge/jobs/{j['id']}/fail", headers=H,
                      json={'error': (r.stderr or '')[-500:]}, timeout=30)
print('processed', done, 'jobs', flush=True)


## Benchmark each T4 separately
Writes a report (seconds, peak VRAM, failures). Run once per GPU.

In [ ]:
!python scripts/ai_bench.py --gpu 1 --models triposr,sf3d --image /kaggle/working/result.glb 2>/dev/null || true
!python scripts/ai_bench.py --list